# Prototipo del Data Agent di SupplyBot

Notebook per **prototipare e testare il pandas agent** prima di integrarlo nell'architettura a microservizi,
come suggerito dalla traccia. L'agente gira in modalità *verbose*: per ogni domanda si vedono
il **pensiero** (cosa vuole calcolare), l'**azione** (il codice che scrive) e l'**osservazione** (il risultato o l'errore).

> Dati e documenti sono **simulati** a scopo didattico; il caso è ispirato ad Atisale S.p.A. solo per i fatti pubblici.

**Come usarlo:** apri la cartella del progetto in VS Code, assicurati che il file `.env` nella radice contenga
`OPENAI_API_KEY`, poi esegui le celle in ordine con il kernel Python dell'ambiente `data_agent`.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# il notebook vive in data_agent/: il .env sta nella cartella superiore
load_dotenv(Path.cwd().parent / ".env")
print("Chiave OpenAI caricata:", bool(os.getenv("OPENAI_API_KEY")))  # non stampiamo mai la chiave
print("Modello:", os.getenv("OPENAI_MODEL_DATA") or os.getenv("OPENAI_MODEL", "gpt-6-luna"))

## 1. Validazione e pulizia dei dati

Il CSV grezzo è volutamente "sporco". Nel servizio la catena gira **una sola volta, all'avvio** (`dataset.py`):
**validazione dello schema** (`schema.py`: colonne, tipi, valori ammessi, formati), **diagnosi** (`profila_dataset`),
**scelta delle pulizie** in base alla diagnosi, **pulizia** e **validazione rigida** del risultato.
Qui la eseguiamo passo per passo. Ogni operazione è scritta in codice testato e ogni correzione viene contata.

In [ ]:
import json
from pulizia import carica_grezzo, profila_dataset, pulisci_dati
from schema import valida_grezzo, valida_pulito
from dataset import scegli_operazioni

grezzo = carica_grezzo()
# 1a. validazione dello schema: errori = file da rifiutare, avvisi = anomalie che la pulizia corregge
print(json.dumps(valida_grezzo(grezzo), indent=1, ensure_ascii=False))
# 1b. diagnosi: da qui si decide quali operazioni applicare
profilo = profila_dataset(grezzo)
print(json.dumps(profilo, indent=1, ensure_ascii=False))

In [ ]:
# 1c. pulizia con le operazioni scelte in base alla diagnosi, poi validazione rigida del risultato
operazioni = scegli_operazioni(profilo)
print("Operazioni scelte:", operazioni)
df, report = pulisci_dati(grezzo, operazioni)
for voce, valore in report.items():
    print(f"{voce}: {valore}")
print("Validazione dopo la pulizia:", valida_pulito(df))

In [ ]:
# Perché le date si leggono un formato alla volta: con un parser unico "giorno prima"
# la data ISO 2025-01-07 diventa il 1° luglio e il sale da disgelo sembra venduto d'estate.
import pandas as pd
esempio = pd.Series(["2025-01-07", "07/01/2025"])
print("Parser unico dayfirst :", pd.to_datetime(esempio, format="mixed", dayfirst=True).dt.date.tolist())
from pulizia import _date_miste
print("Un formato alla volta :", [d.date() for d in _date_miste(esempio)[0]])

## 2. La sandbox: il codice dell'LLM è controllato prima di essere eseguito

Il codice generato gira in un processo separato, con un tempo massimo e senza accesso a file, rete o sistema operativo.

In [ ]:
import tempfile
from sandbox import esegui

cartella = tempfile.mkdtemp()
percorso_df = str(Path(cartella) / "df.pkl")
df.to_pickle(percorso_df)

prove = {
    "codice lecito": "print(df.groupby('site')['qty_t'].sum().round(1))",
    "accesso al sistema": "import os; os.system('dir')",
    "scrittura di file": "df.to_csv('copia.csv')",
    "errore di colonna": "print(df['vettore'].unique())",
}
for nome, codice in prove.items():
    esito = esegui(codice, percorso_df, cartella)
    print(f"--- {nome}: ok={esito['ok']}")
    print(esito["output"] or esito["errore"])

## 3. L'agente in modalità verbose

Ogni domanda attraversa il ciclo ReAct. Annota qui i comportamenti strani: diventano le "sfide tecniche" della presentazione.

In [ ]:
from IPython.display import Image, Markdown, display
from agente_dati import DataAgent, CARTELLA_GRAFICI

agente = DataAgent(verbose=True)

def chiedi(domanda):
    risultato = agente.analizza(domanda)
    print("\n" + "=" * 80)
    display(Markdown("**Sintesi per il manager:**\n\n" + (risultato["sintesi"] or "")))
    for nome in risultato["grafici"]:
        display(Image(filename=str(CARTELLA_GRAFICI / nome)))
    print("Passi eseguiti:", [(p["tipo"], p.get("ok", True)) for p in risultato["passi"]])
    return risultato

In [ ]:
r1 = chiedi("Come sono andati i volumi di sale da disgelo nell'inverno 2025-26, mese per mese?")

In [ ]:
r2 = chiedi("Quali vettori sono sotto l'obiettivo OTIF del 95% nel 2026?")

In [ ]:
r3 = chiedi("Nelle urgenze neve abbiamo rispettato le 48 ore? L'obiettivo è il 90% delle urgenze.")

In [ ]:
r4 = chiedi("Le spedizioni via nave da Sant'Antioco sono meno puntuali d'inverno (dicembre-febbraio) rispetto al resto dell'anno?")

In [ ]:
r5 = chiedi("Quanto sale da disgelo serve a scorta entro il 31 ottobre 2026? La regola è il 50% delle tonnellate spedite nella stagione invernale precedente.")

## 4. Note per la presentazione

- Domande in cui l'agente ha sbagliato al primo tentativo e si è corretto da solo:
- Domande in cui ha risposto senza calcolare (bloccate dalla guardia anti-allucinazione):
- Tempi medi di risposta: